In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_clean = pd.read_csv('Enriched_Flight_Visa_Data.csv')
df_clean = df_clean.dropna(subset=['GDP_pc', 'Population', 'Visa_Acceptance_Rate', 'Outbound_Per_Capita', 'GDP_Cluster'])

sns.set_theme(style="whitegrid")
# Define a consistent color palette for the 4 tiers
tier_palette = {'Highest 25%': 'darkblue', 'Lower High 50-75%': 'royalblue', 'Higher Low 25-50%': 'darkorange', 'Lowest 25%': 'firebrick'}
cluster_order = ['Highest 25%', 'Lower High 50-75%', 'Higher Low 25-50%', 'Lowest 25%']

# GRAPH 1: Scatter Plot colored by GDP Cluster
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=df_clean, x='Visa_Acceptance_Rate', y='Outbound_Per_Capita',
    hue='GDP_Cluster', hue_order=cluster_order, palette=tier_palette,
    size='Population', sizes=(50, 400), alpha=0.7
)
sns.regplot(data=df_clean, x='Visa_Acceptance_Rate', y='Outbound_Per_Capita', scatter=False, color='black', line_kws={'linestyle':'--'})
plt.title('Visa Acceptance Rates vs. Traffic (Segmented by Economic Tier)', fontsize=16)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, title="GDP Tier")
plt.tight_layout()
plt.savefig('visa_vs_traffic_by_tier.png')
plt.show()

# GRAPH 2: 2x2 Dashboard Grouped by GDP_Cluster
# Calculate the average values for each economic tier per year
tier_trends = df_clean.groupby(['Year', 'GDP_Cluster']).agg({
    'Outbound_Per_Capita': 'mean',
    'Inbound_Per_Capita': 'mean',
    'Visa_Acceptance_Rate': 'mean',
    'Population': 'mean'
}).reset_index()

fig, axs = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Macroeconomic & Policy Timelines by Economic Tier (2016-2024)', fontsize=20, fontweight='bold', y=0.98)

# Top Left: Outbound Flights
sns.lineplot(data=tier_trends, x='Year', y='Outbound_Per_Capita', hue='GDP_Cluster', hue_order=cluster_order, palette=tier_palette, marker='o', linewidth=3, ax=axs[0, 0])
axs[0, 0].set_title('Outbound Flights per Capita', fontsize=14, fontweight='bold')
axs[0, 0].legend(title='Economic Tier')

# Top Right: Visa Acceptance Rates
sns.lineplot(data=tier_trends, x='Year', y='Visa_Acceptance_Rate', hue='GDP_Cluster', hue_order=cluster_order, palette=tier_palette, marker='s', linewidth=3, ax=axs[0, 1], legend=False)
axs[0, 1].set_title('Schengen Visa Acceptance Rates (%)', fontsize=14, fontweight='bold')

# Bottom Left: Inbound Flights
sns.lineplot(data=tier_trends, x='Year', y='Inbound_Per_Capita', hue='GDP_Cluster', hue_order=cluster_order, palette=tier_palette, marker='^', linewidth=3, ax=axs[1, 0], legend=False)
axs[1, 0].set_title('Inbound Flights per Capita', fontsize=14, fontweight='bold')

# Bottom Right: Population
sns.lineplot(data=tier_trends, x='Year', y='Population', hue='GDP_Cluster', hue_order=cluster_order, palette=tier_palette, marker='D', linewidth=3, ax=axs[1, 1], legend=False)
axs[1, 1].set_title('Average Population', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.savefig('timeline_by_gdp_tier.png', dpi=300)
plt.show()